# Pricing an Equity Index Future (S&P 500)

This notebook prices an **equity index future** (e.g. an S&P 500 / E-mini–style contract)
by **reusing the Monte Carlo library** from
[`AnonymousJY/mkt-depth-n-resiliency`](https://github.com/AnonymousJY/mkt-depth-n-resiliency).

We don't write any new pricing logic — we just wire together components that already exist
in `Library/`:

| Component | Class used | Role |
|---|---|---|
| Payoff | `PayoffForward` (via `payoff_mc_factory("forward")`) | \(S_T - K\) |
| Product | `PathDependentAsianDiscrete` (single fixing = delivery) | terminal-spot European product |
| Model / paths | `ExoticEngineBlackScholesMerton` | risk-neutral GBM with carry \(r-q-b\) |
| RNG | `RandomMT19937` | Mersenne-Twister draws |
| Statistics | `StatisticsMCMean` | averages discounted payoffs |

**Pricing idea (full cost-of-carry).** Under the risk-neutral measure the *fair
futures/forward price* is the expected terminal spot. With a continuous dividend yield
\(q\) **and** a securities-lending / borrow (repo) rate \(b\) earned by the holder of the
shares,
$$F = \mathbb{E}^{\mathbb{Q}}[S_T] = S_0\, e^{(r-q-b)T}.$$
The borrow rate enters with the **same sign as dividends**: lending out the stock you are
long earns the fee \(b\), which cheapens the carry and *lowers* the fair future. For a
liquid index \(b\) is small but generally nonzero (the index repo / funding basis); for
hard-to-borrow single names it can be large ("specialness").

**Reuse trick.** The BSM engine's risk-neutral drift is \(r - (\text{dividend yield})\),
so we simply feed it an **effective yield \(q+b\)**, producing the drift \(r-q-b\)
exactly — no change to the library. Discounting still uses \(r\). The product machinery
returns the discounted payoff \(e^{-rT}(S_T-K)\); with \(K=0\) and **undoing the discount
factor** we recover \(\mathbb{E}^{\mathbb{Q}}[S_T]\), i.e. the fair future price, which we
cross-check against the closed form.

In [1]:
import os, sys
import numpy as np

# --- Point this at your local clone of mkt-depth-n-resiliency -------------------
# The repo root is the folder that contains the `Library/` package.
# Edit REPO_ROOT if auto-detection below does not find it.
REPO_ROOT = None
_here = os.getcwd(); _cands = [_here]
for _ in range(5):
    _here = os.path.abspath(os.path.join(_here, os.pardir)); _cands.append(_here)
_cands.append(os.path.expanduser("~/mkt-depth-n-resiliency"))
for cand in _cands:
    if cand and os.path.isdir(os.path.join(cand, "Library")):
        REPO_ROOT = cand
        break
assert REPO_ROOT, "Set REPO_ROOT to the folder containing Library/"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Using repo at:", REPO_ROOT)

from Library.Parameters import ParametersConstant
from Library.PayoffFactory import payoff_mc_factory
from Library.PathDependent import PathDependentAsianDiscrete
from Library.ExoticEngine import ExoticEngineBlackScholesMerton
from Library.StatisticsMC import StatisticsMCMean
from Library.Random import RandomMT19937

Using repo at: /tmp/repo


## Market inputs

Sensible S&P 500 defaults — edit any of these. Volatility does **not** affect the fair
future price (the expectation of \(S_T\) is drift-only); it is supplied because the BSM
engine requires it and it does drive the Monte Carlo standard error.

`b` is the **annual borrow / securities-lending (repo) rate**. Set `b = 0.0` to recover the
plain \(r-q\) carry. The default below is a small *illustrative* index repo spread — replace
it with your desk's implied financing basis.

In [2]:
S0    = 5500.0   # current S&P 500 index level
r     = 0.043    # continuously-compounded risk-free rate (annual)
q     = 0.013    # continuous dividend yield of the index (annual)
b     = 0.0025   # borrow / securities-lending (repo) rate (annual)  <-- illustrative; set 0.0 for none
T     = 0.25     # time to delivery, in years (~3 months)
sigma = 0.20     # index volatility (only affects MC noise, not the fair price)

N_PATHS = 200_000
SEED    = 42
MULTIPLIER = 50  # S&P 500 E-mini contract multiplier ($50 x index); set 250 for the big contract

## Build the contract from library components

A future on the terminal index level is a `PathDependentAsianDiscrete` with a **single
fixing** at delivery (so the "average" equals \(S_T\)) wrapped around a `PayoffForward`
with strike 0.

The borrow rate is folded into an **effective dividend yield** `q + b` handed to the engine,
so its drift becomes \(r - q - b\).

In [3]:
rfr = ParametersConstant(np.array(r))          # risk-free rate term structure (flat)
div = ParametersConstant(np.array(q + b))      # EFFECTIVE yield = dividends + borrow  -> drift r-q-b
vol = ParametersConstant(np.array(sigma))      # volatility (flat)

# PayoffForward(strike=0)  ->  payoff = S_T - 0 = S_T
forward_payoff = payoff_mc_factory("forward")(strike=np.array(0.0))

product = PathDependentAsianDiscrete(
    fixing_times=np.array([T]),     # single fixing at delivery => average == S_T
    delivery_time=np.array(T),
    the_payoff=forward_payoff,
    quantity_amount=np.array(1.0),
)

rng = RandomMT19937(seed=SEED)
engine = ExoticEngineBlackScholesMerton(
    the_product=product,
    risk_free_rate=rfr,
    dividend_yield=[div],
    imp_volatility=[vol],
    rand_generator=rng,
    spot_price=np.array(S0),
    number_of_paths=np.uint64(N_PATHS),
)
print("Engine and product built. Effective carry rate r-q-b = %.4f" % (r - q - b))

Engine and product built. Effective carry rate r-q-b = 0.0275


## Run the Monte Carlo and recover the fair future price

`do_simulation` fills the mean gatherer with **discounted** per-path payoffs
\(e^{-rT}S_T\). Dividing the average by the discount factor gives
\(\mathbb{E}^{\mathbb{Q}}[S_T]=F\). Because `StatisticsMCMean` keeps the full per-path
vector, we can also read off a Monte Carlo standard error / 95% CI directly.

In [4]:
gatherer = StatisticsMCMean()
engine.do_simulation(gatherer)

disc = float(np.exp(-r * T))                        # e^{-rT}
mc_discounted_mean = gatherer.get_result_so_far().item()
fair_future_mc = mc_discounted_mean / disc          # undo discount -> E[S_T]

# Per-path samples (discounted) are stored on the gatherer; undiscount for the price dist.
samples = np.asarray(gatherer.running_sum).reshape(-1) / disc
se = samples.std(ddof=1) / np.sqrt(samples.size)
ci_lo, ci_hi = fair_future_mc - 1.96 * se, fair_future_mc + 1.96 * se

print(f"Fair future price (MC)   : {fair_future_mc:,.4f}")
print(f"MC standard error        : {se:,.4f}")
print(f"95% confidence interval  : [{ci_lo:,.4f}, {ci_hi:,.4f}]")

Fair future price (MC)   : 5,537.6707
MC standard error        : 1.2431
95% confidence interval  : [5,535.2341, 5,540.1072]


## Verification: analytic cost-of-carry (with borrow)

Closed form \(F = S_0 e^{(r-q-b)T}\). The Monte Carlo estimate should agree to within a few
standard errors, and the analytic value should fall inside the 95% CI.

In [5]:
fair_future_analytic = S0 * np.exp((r - q - b) * T)
abs_err = abs(fair_future_mc - fair_future_analytic)
rel_err = abs_err / fair_future_analytic
in_ci   = ci_lo <= fair_future_analytic <= ci_hi

print(f"Fair future price (analytic): {fair_future_analytic:,.4f}")
print(f"Fair future price (MC)      : {fair_future_mc:,.4f}")
print(f"Absolute error              : {abs_err:,.4f}")
print(f"Relative error              : {rel_err:.3e}")
print(f"Analytic value inside 95% CI: {bool(in_ci)}")
assert in_ci, "MC and analytic disagree beyond Monte Carlo error!"
print("\nNotional per contract       : {:,.2f}".format(fair_future_mc * MULTIPLIER))

Fair future price (analytic): 5,537.9428
Fair future price (MC)      : 5,537.6707
Absolute error              : 0.2721
Relative error              : 4.914e-05
Analytic value inside 95% CI: True

Notional per contract       : 276,883.53


## Impact of the borrow rate

How much does the borrow assumption move the fair future? Each basis point of borrow shifts
\(F\) by roughly \(-S_0 \cdot T \cdot 1\text{bp}\).

In [6]:
def fair_price_analytic(b_):
    return S0 * np.exp((r - q - b_) * T)

base = fair_price_analytic(0.0)
print(f"{'borrow b':>10} | {'fair F':>12} | {'vs b=0':>10}")
print("-" * 38)
for b_ in [0.0, 0.0025, 0.005, 0.01, 0.02]:
    f_ = fair_price_analytic(b_)
    print(f"{b_:10.4f} | {f_:12,.4f} | {f_-base:10,.4f}")

  borrow b |       fair F |     vs b=0
--------------------------------------
    0.0000 |   5,541.4051 |     0.0000
    0.0025 |   5,537.9428 |    -3.4623
    0.0050 |   5,534.4826 |    -6.9224
    0.0100 |   5,527.5689 |   -13.8362
    0.0200 |   5,513.7672 |   -27.6379


## Reusable helper

A thin wrapper so you can reprice for any inputs, now including the borrow rate `b`. Returns
the MC fair price, the analytic cost-of-carry benchmark, and the MC standard error.

In [7]:
def price_equity_future(S0, r, q, T, b=0.0, sigma=0.20, n_paths=200_000, seed=42, repo_root=REPO_ROOT):
    """Fair price of an equity index future via the mkt-depth-n-resiliency MC library.

    Carry rate is r - q - b (b = borrow / securities-lending rate). Returns a dict with the
    MC price, the analytic cost-of-carry price, and the MC standard error.
    """
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    rfr = ParametersConstant(np.array(r))
    div = ParametersConstant(np.array(q + b))           # effective yield = dividends + borrow
    vol = ParametersConstant(np.array(sigma))
    payoff = payoff_mc_factory("forward")(strike=np.array(0.0))
    product = PathDependentAsianDiscrete(
        fixing_times=np.array([T]), delivery_time=np.array(T),
        the_payoff=payoff, quantity_amount=np.array(1.0))
    engine = ExoticEngineBlackScholesMerton(
        the_product=product, risk_free_rate=rfr, dividend_yield=[div],
        imp_volatility=[vol], rand_generator=RandomMT19937(seed=seed),
        spot_price=np.array(S0), number_of_paths=np.uint64(n_paths))
    g = StatisticsMCMean()
    engine.do_simulation(g)
    disc = float(np.exp(-r * T))
    mc = g.get_result_so_far().item() / disc
    se = float(np.asarray(g.running_sum).reshape(-1).std(ddof=1) / np.sqrt(n_paths)) / disc
    return {"mc": mc, "analytic": float(S0 * np.exp((r - q - b) * T)), "se": se}

# Term structure of fair futures prices across maturities (using the borrow rate b above)
print(f"{'T (yrs)':>8} | {'MC fair':>12} | {'Analytic':>12} | {'MC SE':>8}")
print("-" * 50)
for t in [1/12, 0.25, 0.5, 0.75, 1.0]:
    res = price_equity_future(S0, r, q, t, b=b)
    print(f"{t:8.4f} | {res['mc']:12,.4f} | {res['analytic']:12,.4f} | {res['se']:8.4f}")

 T (yrs) |      MC fair |     Analytic |    MC SE
--------------------------------------------------
  0.0833 |   5,512.4431 |   5,512.6186 |   0.7133
  0.2500 |   5,537.6707 |   5,537.9428 |   1.2431
  0.5000 |   5,575.8061 |   5,576.1473 |   1.7746
  0.7500 |   5,614.2383 |   5,614.6154 |   2.1940
  1.0000 |   5,652.9533 |   5,653.3489 |   2.5573


## Implied carry: invert the model to match a market quote

This is how the cost-of-carry model is most often used in practice. Instead of *computing* a
fair price from assumed inputs, we take the **market-quoted future price** and **solve for the
carry rate the market is pricing in**.

Because the fair price is monotically increasing in the total carry rate
\(c = r - q - b\) (and decreasing in \(b\)), we solve for \(c\) using the repo's own
`bisection` routine (`Library/RootFinder.py`) on the MC-validated closed form
\(F = S_0 e^{cT}\). We then split \(c\) into an **implied borrow** (holding the dividend
yield fixed) or an **implied dividend yield** (holding borrow fixed). The gap between this
implied carry and your assumed \(r-q-b\) *is* the basis.

In [8]:
from Library.RootFinder import bisection

# --- Market-quoted future price (edit to your screen) -------------------------
market_future = 5520.0

# Fair future as a function of total carry c = r - q - b. Monotonic & increasing in c, so the
# repo's bisection applies directly. The MC engine validated F = S0*exp(c*T) above.
def future_from_carry(c):
    return S0 * np.exp(np.asarray(c) * T)

implied_carry = bisection(
    func=future_from_carry,
    lower_value=np.array([-0.5]),     # wide bracket on the carry rate
    upper_value=np.array([0.5]),
    target_value=np.array([float(market_future)]),
    initial_value=np.array([0.0]),
).item()

# Decompose c = r - q - b two ways:
implied_b = r - q - implied_carry     # implied borrow/repo, holding dividend yield q fixed
implied_q = r - b - implied_carry     # implied dividend yield, holding borrow b fixed
basis_vs_model = market_future - fair_future_analytic

print(f"Market future price            : {market_future:,.4f}")
print(f"Model fair price (assumed inputs): {fair_future_analytic:,.4f}")
print(f"Basis (market - model)         : {basis_vs_model:+,.4f}")
print("-" * 48)
print(f"Implied total carry (r-q-b)    : {implied_carry:.4%}")
print(f"  implied borrow b  (q={q:.2%} fixed): {implied_b:.4%}")
print(f"  implied div yield q (b={b:.2%} fixed): {implied_q:.4%}")
print(f"Reprice check  S0*exp(carry*T) : {future_from_carry(implied_carry).item():,.4f}")

Market future price            : 5,520.0000
Model fair price (assumed inputs): 5,537.9428
Basis (market - model)         : -17.9428
------------------------------------------------
Implied total carry (r-q-b)    : 1.4519%
  implied borrow b  (q=1.30% fixed): 1.5481%
  implied div yield q (b=0.25% fixed): 2.5981%
Reprice check  S0*exp(carry*T) : 5,520.0000


In [9]:
def imply_carry(market_future, S0, r, q, T, b_assumed=0.0):
    """Invert the cost-of-carry model with the repo's bisection root-finder.

    Returns the implied total carry c = r-q-b, the implied borrow (given q), and the
    implied dividend yield (given b_assumed).
    """
    f = lambda c: S0 * np.exp(np.asarray(c) * T)
    c = bisection(func=f,
                  lower_value=np.array([-0.5]), upper_value=np.array([0.5]),
                  target_value=np.array([float(market_future)]),
                  initial_value=np.array([0.0])).item()
    return {"implied_carry": c,
            "implied_borrow_given_q": r - q - c,
            "implied_div_given_b": r - b_assumed - c}

# Implied borrow across a range of hypothetical market quotes (q held fixed):
print(f"{'mkt future':>11} | {'impl carry':>11} | {'impl borrow b':>14}")
print("-" * 42)
for mkt in [5510, 5520, 5530, 5541.41, 5550]:
    res = imply_carry(mkt, S0, r, q, T, b_assumed=b)
    print(f"{mkt:11,.2f} | {res['implied_carry']:11.4%} | {res['implied_borrow_given_q']:14.4%}")

 mkt future |  impl carry |  impl borrow b
------------------------------------------
   5,510.00 |     0.7266% |        2.2734%
   5,520.00 |     1.4519% |        1.5481%
   5,530.00 |     2.1759% |        0.8241%
   5,541.41 |     3.0004% |       -0.0004%
   5,550.00 |     3.6199% |       -0.6199%


## Analytic (closed-form) pricer

A future has **no optionality**, so its price is a deterministic carry relationship — there is
an exact closed form and no integral to approximate. `Library/FuturePricerAnalytic.py` (written in the same vectorised style as
`Library/OptionPricerBSM1973.py`, living alongside the repo's other pricers) provides:

| Function | Formula |
|---|---|
| `future_fair_price` | \(F = S_0 e^{(r-q-b)T}\) |
| `forward_value` | \(V = S_0 e^{-(q+b)T} - K e^{-rT}\) (PV of a long contract struck at \(K\)) |
| `forward_delta` | \(\partial V/\partial S_0 = e^{-(q+b)T}\) |
| `forward_rho` | \(\partial V/\partial r = K T e^{-rT}\) |
| `forward_theta` | \(\partial V/\partial T\) |
| `implied_carry_from_price` | \(c = \ln(F/S_0)/T\) (closed-form carry inversion) |

Below we price both the **fair price** and the **mark-to-market value** of an existing long
position struck at \(K\), and confirm each matches the Monte Carlo engine.

In [10]:
from Library.FuturePricerAnalytic import (future_fair_price, forward_value, forward_delta,
                                          forward_rho, forward_theta, implied_carry_from_price)

K = 5400.0   # delivery price LOCKED IN at trade inception for an existing long position.
             # NOT today's screen quote -- the screen quote is the current fair price F
             # (= fair_future_analytic / market_future). Value V = e^{-rT}(F - K);
             # set K = F for a brand-new trade (value 0).

a = lambda x: np.array(x)
F_an = future_fair_price(a(S0), a(r), a(q), a(b), a(T)).item()
V_an = forward_value(a(S0), a(K), a(r), a(q), a(b), a(T)).item()
delta = forward_delta(a(r), a(q), a(b), a(T)).item()
rho   = forward_rho(a(K), a(r), a(T)).item()
theta = forward_theta(a(S0), a(K), a(r), a(q), a(b), a(T)).item()

print("ANALYTIC")
print(f"  fair (par) price F        : {F_an:,.4f}")
print(f"  value of long @ K={K:,.0f}   : {V_an:,.4f}")
print(f"  delta (dV/dS0)            : {delta:.6f}")
print(f"  rho   (dV/dr)             : {rho:,.4f}")
print(f"  theta (dV/dT)             : {theta:,.4f}")

ANALYTIC
  fair (par) price F        : 5,537.9428
  value of long @ K=5,400   : 136.4678
  delta (dV/dS0)            : 0.996132
  rho   (dV/dr)             : 1,335.5652
  theta (dV/dT)             : 144.7969


In [11]:
# Cross-check the analytic CONTRACT VALUE against the MC engine (PayoffForward struck at K).
# Here the engine's discounted payoff IS the contract value, so we do NOT undo the discount.
payoff_K = payoff_mc_factory("forward")(strike=np.array(K))
product_K = PathDependentAsianDiscrete(
    fixing_times=np.array([T]), delivery_time=np.array(T),
    the_payoff=payoff_K, quantity_amount=np.array(1.0))
engine_K = ExoticEngineBlackScholesMerton(
    the_product=product_K, risk_free_rate=ParametersConstant(np.array(r)),
    dividend_yield=[ParametersConstant(np.array(q + b))], imp_volatility=[ParametersConstant(np.array(sigma))],
    rand_generator=RandomMT19937(seed=SEED), spot_price=np.array(S0), number_of_paths=np.uint64(N_PATHS))
g_K = StatisticsMCMean(); engine_K.do_simulation(g_K)
V_mc = g_K.get_result_so_far().item()
se_K = np.asarray(g_K.running_sum).reshape(-1).std(ddof=1) / np.sqrt(N_PATHS)

print(f"{'':22}{'analytic':>12}{'MC':>14}{'MC 95% CI':>26}")
print(f"{'fair price F':22}{F_an:12,.4f}{fair_future_mc:14,.4f}{'[%.2f, %.2f]'%(ci_lo,ci_hi):>26}")
print(f"{'value @ K=%.0f'%K:22}{V_an:12,.4f}{V_mc:14,.4f}{'[%.2f, %.2f]'%(V_mc-1.96*se_K, V_mc+1.96*se_K):>26}")
assert abs(V_an - V_mc) < 5*se_K, "analytic vs MC mismatch beyond MC error"
print("\nAnalytic and Monte Carlo agree within Monte Carlo error. ✓")

                          analytic            MC                 MC 95% CI
fair price F            5,537.9428    5,537.6707        [5535.23, 5540.11]
value @ K=5400            136.4678      136.1986          [133.79, 138.61]

Analytic and Monte Carlo agree within Monte Carlo error. ✓


### Daily settlement: how a future's value resets to zero

A listed future is marked to market each day. Entering at the settlement price means strike
\(K=F\) so \(V=0\); a day later spot has moved and the position carries value, paid/collected
as **variation margin**; the exchange then **re-strikes \(K\) to the new settlement**, zeroing
it again. *(Settlement is the futures price; we use the fair forward as a sub-bp proxy and
advance time by one day.)*

In [12]:
# --- Daily settlement cycle -----------------------------------------------------
day_return = 0.01          # illustrative one-day index move (+1%)
dt_day     = 1 / 365

# Day 0: enter long at the prevailing fair (settlement) price -> strike K = F0 -> value 0
F0 = future_fair_price(a(S0), a(r), a(q), a(b), a(T)).item()
K0 = F0
V0 = forward_value(a(S0), a(K0), a(r), a(q), a(b), a(T)).item()
print(f"Day 0:  settle F0 = {F0:,.4f}   strike K = F0   ->  value = {V0:,.6f}")

# Day 1: one day passes (T shrinks) and spot moves
T1 = T - dt_day
S1 = S0 * (1 + day_return)
F1 = future_fair_price(a(S1), a(r), a(q), a(b), a(T1)).item()
V1 = forward_value(a(S1), a(K0), a(r), a(q), a(b), a(T1)).item()   # MtM of long still struck at K0
vm = (F1 - F0) * MULTIPLIER                                        # futures variation margin (undiscounted)
print(f"Day 1:  spot {S0:,.0f} -> {S1:,.0f} ({day_return:+.1%}),  settle F1 = {F1:,.4f}")
print(f"        mark-to-market value (struck at K0): {V1:,.4f} pts  =  ${V1*MULTIPLIER:,.2f}")
print(f"        futures variation margin (F1-F0)xM : ${vm:,.2f}   (undiscounted; ~1-day discount apart)")

# Re-strike to the new settlement price -> value back to zero
K1 = F1
V1_reset = forward_value(a(S1), a(K1), a(r), a(q), a(b), a(T1)).item()
print(f"        re-strike K -> F1 = {K1:,.4f}   ->  value = {V1_reset:,.6f}   (reset to zero)")
assert abs(V1_reset) < 1e-9
print("\nAfter re-striking to settlement, the carried value is zero again. ✓")

Day 0:  settle F0 = 5,537.9428   strike K = F0   ->  value = 0.000000


Day 1:  spot 5,500 -> 5,555 (+1.0%),  settle F1 = 5,592.9008
        mark-to-market value (struck at K0): 54.3768 pts  =  $2,718.84
        futures variation margin (F1-F0)xM : $2,747.90   (undiscounted; ~1-day discount apart)
        re-strike K -> F1 = 5,592.9008   ->  value = 0.000000   (reset to zero)

After re-striking to settlement, the carried value is zero again. ✓


## Risk: equity delta, IR01, dividend (Div01)

Risk of the **long position struck at \(K\)**, with \(V = S_0 e^{-(q+b)T} - K e^{-rT}\):

| Greek | Closed form | Meaning |
|---|---|---|
| **Equity delta** \(\partial V/\partial S_0\) | \(e^{-(q+b)T}\) | per index point (×multiplier → \$ per point) |
| **IR01** | \(K T e^{-rT}\cdot 10^{-4}\) | value change for **+1bp** in \(r\) |
| **Div01** | \(-S_0 T e^{-(q+b)T}\cdot 10^{-4}\) | value change for **+1bp** in dividend yield |

IR01 is positive (a higher rate discounts the fixed delivery price \(K\) more, helping the
long), while Div01 is negative (richer dividends lower the forward). The borrow sensitivity is
identical to Div01 since \(b\) and \(q\) enter together. We verify each against a
finite-difference bump-and-revalue on `forward_value`.

In [13]:
from Library.FuturePricerAnalytic import forward_ir01, forward_dividend_rho, forward_div01

# Analytic greeks for the long position struck at K (from the analytic section above)
eq_delta = forward_delta(a(r), a(q), a(b), a(T)).item()         # per index point
ir01     = forward_ir01(a(K), a(r), a(T)).item()                # per +1bp in r
div01    = forward_div01(a(S0), a(q), a(b), a(T)).item()        # per +1bp in dividend yield

# Dollar figures per contract
eq_delta_usd = eq_delta * MULTIPLIER          # $ per 1 index-point move
ir01_usd     = ir01 * MULTIPLIER              # $ per +1bp in r
div01_usd    = div01 * MULTIPLIER             # $ per +1bp in dividends

print(f"{'Greek':<16}{'index pts':>14}{'$ / contract':>16}")
print("-" * 46)
print(f"{'Equity delta':<16}{eq_delta:>14.6f}{eq_delta_usd:>16,.2f}")
print(f"{'IR01 (+1bp r)':<16}{ir01:>14.6f}{ir01_usd:>16,.4f}")
print(f"{'Div01 (+1bp q)':<16}{div01:>14.6f}{div01_usd:>16,.4f}")
print(f"\n(borrow01 = Div01 = {div01:.6f}; equity dollar-delta per 1% move = "
      f"{eq_delta*MULTIPLIER*S0*0.01:,.2f})")

Greek                index pts    $ / contract
----------------------------------------------
Equity delta          0.996132           49.81
IR01 (+1bp r)         0.133557          6.6778
Div01 (+1bp q)       -0.136968         -6.8484

(borrow01 = Div01 = -0.136968; equity dollar-delta per 1% move = 2,739.36)


In [14]:
# --- Finite-difference verification (bump-and-revalue on forward_value) -------------
val = lambda S0_, K_, r_, q_, b_: forward_value(a(S0_), a(K_), a(r_), a(q_), a(b_), a(T)).item()
V0  = val(S0, K, r, q, b)
h_s = 1e-4 * S0
bp  = 1e-4

delta_fd = (val(S0 + h_s, K, r, q, b) - val(S0 - h_s, K, r, q, b)) / (2 * h_s)  # dV/dS0
ir01_fd  = val(S0, K, r + bp, q, b) - V0                                        # +1bp in r
div01_fd = val(S0, K, r, q + bp, b) - V0                                        # +1bp in q

print(f"{'Greek':<16}{'analytic':>14}{'finite-diff':>16}{'abs diff':>14}")
print("-" * 60)
print(f"{'Equity delta':<16}{eq_delta:>14.8f}{delta_fd:>16.8f}{abs(eq_delta-delta_fd):>14.2e}")
print(f"{'IR01':<16}{ir01:>14.8f}{ir01_fd:>16.8f}{abs(ir01-ir01_fd):>14.2e}")
print(f"{'Div01':<16}{div01:>14.8f}{div01_fd:>16.8f}{abs(div01-div01_fd):>14.2e}")
for nm, an, fd in [("delta",eq_delta,delta_fd),("ir01",ir01,ir01_fd),("div01",div01,div01_fd)]:
    assert abs(an-fd) < 1e-4*max(1,abs(an)) + 1e-6, f"{nm} mismatch"
print("\nAll greeks match finite differences. ✓")

Greek                 analytic     finite-diff      abs diff
------------------------------------------------------------
Equity delta        0.99613250      0.99613250      3.80e-13
IR01                0.13355652      0.13355485      1.67e-06
Div01              -0.13696822     -0.13696651      1.71e-06

All greeks match finite differences. ✓


### Two deltas: futures-price delta (> 1) vs value delta (< 1)

The delta of a future is **above 1** if you mean the sensitivity of the futures *price* to
spot, and **below 1** if you mean the sensitivity of the contract's present *value*:

$$\frac{\partial F}{\partial S_0} = e^{(r-q-b)T} \;>\; 1 \iff r-q-b>0,
\qquad
\frac{\partial V}{\partial S_0} = e^{-(q+b)T} \;<\; 1,
\qquad
\frac{\partial F}{\partial S_0} = e^{rT}\,\frac{\partial V}{\partial S_0}.$$

The price isn't discounted (it's paid at delivery), so its spot-sensitivity is scaled up by
\(e^{rT}\) and exceeds 1 when net carry is positive — the usual case for index futures since
\(r>q\). *(With borrow as a financing cost, F = S_0 e^{(r+b-q)T}, the same condition reads
\(r+b>q\).)*

In [15]:
from Library.FuturePricerAnalytic import future_price_delta

price_delta = future_price_delta(a(r), a(q), a(b), a(T)).item()   # dF/dS0
value_delta = forward_delta(a(r), a(q), a(b), a(T)).item()        # dV/dS0 (= eq_delta above)

print(f"net carry r-q-b              : {r-q-b:+.4%}")
print(f"futures-PRICE delta dF/dS0   : {price_delta:.6f}   ({'> 1' if price_delta>1 else '<= 1'})")
print(f"contract-VALUE delta dV/dS0  : {value_delta:.6f}   ({'> 1' if value_delta>1 else '< 1'})")
print(f"check  e^(rT) * dV/dS0       : {np.exp(r*T)*value_delta:.6f}")
assert abs(np.exp(r*T)*value_delta - price_delta) < 1e-12
print(f"\nSpot hedge ratio: short {1/price_delta:.6f} index-units of futures exposure per 1 unit of spot")

net carry r-q-b              : +2.7500%
futures-PRICE delta dF/dS0   : 1.006899   (> 1)
contract-VALUE delta dV/dS0  : 0.996132   (< 1)
check  e^(rT) * dV/dS0       : 1.006899

Spot hedge ratio: short 0.993149 index-units of futures exposure per 1 unit of spot


### Dollar greeks

Convert the per-unit greeks to cash by multiplying by the **contract multiplier** \(M\) and
the number of contracts \(N\); for the rate greek also scale by the bump (\(10^{-4}\) for a
"per-bp" figure):

$$\$\Delta = \Delta\cdot M\cdot N,\quad
\text{cash }\Delta = \Delta\cdot M\cdot N\cdot S_0,\quad
\text{IR01}_\$ = \rho\cdot M\cdot N\cdot 10^{-4}.$$

In [16]:
# --- Dollar greeks --------------------------------------------------------------
NUM_CONTRACTS = 1                       # position size (number of futures contracts)
pos = MULTIPLIER * NUM_CONTRACTS        # $ per index point for the whole position

# DELTA -> $
dollar_delta_per_pt   = value_delta * pos          # $ P&L per +1 index point
cash_delta            = value_delta * pos * S0      # notional equity exposure ($)
dollar_delta_per_1pct = cash_delta * 0.01           # $ P&L per +1% index move

# RHO -> $  (rho = dV/dr, in index pts per unit (1.00) change in r)
ir01_dollar         = rho * pos * 1e-4              # $ per +1bp  (IR01 in dollars)
rho_dollar_per_1pct = rho * pos * 0.01             # $ per +1% (100bp)
rho_dollar_per_unit = rho * pos                     # $ per +1.00 (100%) change in r

print(f"position: {NUM_CONTRACTS} contract(s)  x  ${MULTIPLIER}/pt  |  spot {S0:,.0f}")
print("\nDELTA")
print(f"  $ delta per +1 pt     : {dollar_delta_per_pt:>14,.2f}")
print(f"  cash / notional delta : {cash_delta:>14,.2f}")
print(f"  $ delta per +1% move  : {dollar_delta_per_1pct:>14,.2f}")
print("\nRHO")
print(f"  IR01  ($ per +1bp)    : {ir01_dollar:>14,.4f}")
print(f"  $ rho per +1% (100bp) : {rho_dollar_per_1pct:>14,.2f}")
print(f"  $ rho per +1.00 in r  : {rho_dollar_per_unit:>14,.2f}")

position: 1 contract(s)  x  $50/pt  |  spot 5,500

DELTA
  $ delta per +1 pt     :          49.81
  cash / notional delta :     273,936.44
  $ delta per +1% move  :       2,739.36

RHO
  IR01  ($ per +1bp)    :         6.6778
  $ rho per +1% (100bp) :         667.78
  $ rho per +1.00 in r  :      66,778.26


## Futures vs forward: the convexity adjustment

The cost-of-carry price above is really a **forward** price (it assumes deterministic rates).
A **future** is marked-to-market daily, so its price is a martingale under the risk-neutral
measure while the forward lives under the \(T\)-forward measure. For lognormal spot and
Gaussian rates the two are linked by

$$\mathrm{Fut}_0 = \mathrm{Fwd}_0 \cdot e^{\,c},\qquad c = \mathrm{Cov}^{\mathbb Q}\!\Big(\ln S_T,\ \int_0^T r_s\,ds\Big).$$

Under a Hull-White short rate (mean reversion \(a\), vol \(\sigma_r\)) with equity vol
\(\sigma_S\) and equity-rate correlation \(\rho\),

$$c = \rho\,\sigma_S\,\sigma_r\,\frac1a\Big(T-\frac{1-e^{-aT}}{a}\Big)\;\xrightarrow[a\to0]{}\;\rho\,\sigma_S\,\sigma_r\,\frac{T^2}{2}.$$

The sign is that of \(\rho\): positive equity-rate correlation makes the future **richer**
than the forward. The size grows like \(T^2\). For an equity index the effect is tiny — a
fraction of a basis point at a quarter — which is why the forward formula is used in practice.
*(Parameters below are illustrative; calibrate \(\sigma_r,\rho,a\) to your rates model.)*

In [17]:
from Library.FuturePricerAnalytic import futures_convexity_logadj, futures_price_from_forward

# Illustrative Gaussian (Hull-White) rate parameters -- replace with your calibration
sigma_r = 0.01    # short-rate volatility (absolute, annualised) ~ 100 bp/yr
rho_Sr  = 0.20    # equity / short-rate correlation (SIGN MATTERS)
a_mr    = 0.05    # mean-reversion speed

Fwd = fair_future_analytic                      # cost-of-carry price = forward
c   = futures_convexity_logadj(a(sigma), a(sigma_r), a(rho_Sr), a(T), a(a_mr)).item()
Fut = futures_price_from_forward(a(Fwd), a(sigma), a(sigma_r), a(rho_Sr), a(T), a(a_mr)).item()

print(f"Forward price (cost-of-carry): {Fwd:,.4f}")
print(f"Futures price (convex-adj)   : {Fut:,.4f}")
print(f"Basis Fut - Fwd              : {Fut-Fwd:+.4f} index pts  ({c*1e4:+.4f} bp)")

Forward price (cost-of-carry): 5,537.9428
Futures price (convex-adj)   : 5,538.0117
Basis Fut - Fwd              : +0.0689 index pts  (+0.1245 bp)


In [18]:
# Convexity basis grows ~ T^2; negligible short-dated, larger for long maturities.
print(f"{'T (yrs)':>8} | {'forward':>11} | {'futures':>11} | {'basis pts':>10} | {'basis bp':>9}")
print("-" * 60)
for t in [0.25, 0.5, 1.0, 2.0, 5.0]:
    fwd_t = future_fair_price(a(S0), a(r), a(q), a(b), a(t)).item()
    c_t   = futures_convexity_logadj(a(sigma), a(sigma_r), a(rho_Sr), a(t), a(a_mr)).item()
    fut_t = fwd_t * np.exp(c_t)
    print(f"{t:8.2f} | {fwd_t:11,.4f} | {fut_t:11,.4f} | {fut_t-fwd_t:10.4f} | {c_t*1e4:9.4f}")

 T (yrs) |     forward |     futures |  basis pts |  basis bp
------------------------------------------------------------
    0.25 |  5,537.9428 |  5,538.0117 |     0.0689 |    0.1245
    0.50 |  5,576.1473 |  5,576.4238 |     0.2765 |    0.4959
    1.00 |  5,653.3489 |  5,654.4611 |     1.1122 |    1.9671
    2.00 |  5,810.9734 |  5,815.4727 |     4.4994 |    7.7399
    5.00 |  6,310.7094 |  6,339.8570 |    29.1476 |   46.0813


### Convexity-aware implied carry

If the number on your screen is an exchange **futures** quote (not a forward), invert it in two
steps: first strip the convexity to recover the implied **forward**,
\(\mathrm{Fwd}=\mathrm{Fut}\,e^{-c}\), then back out the carry from the forward exactly as
before (reusing `RootFinder.bisection`). Skipping this biases the implied carry by \(\approx
c/T\) — small at the front, larger further out.

The round-trip table is a self-check: feed each maturity's **model** futures price back through
the solver. The convexity-aware inversion recovers the true \(r-q-b\) (here borrow
\(b=0.25\%\)) exactly; the naive inversion (treating the future as a forward) drifts.

In [19]:
# Reinterpret the earlier quote as a FUTURES price and strip convexity before inverting.
c_q          = futures_convexity_logadj(a(sigma), a(sigma_r), a(rho_Sr), a(T), a(a_mr)).item()
implied_fwd  = market_future * np.exp(-c_q)          # Fwd = Fut * exp(-c)
carry_cvx    = bisection(func=future_from_carry,
                         lower_value=np.array([-0.5]), upper_value=np.array([0.5]),
                         target_value=np.array([float(implied_fwd)]),
                         initial_value=np.array([0.0])).item()
b_cvx        = r - q - carry_cvx

print(f"Market quote (as FUTURES)      : {market_future:,.4f}")
print(f"Convexity c                    : {c_q*1e4:+.4f} bp")
print(f"Implied FORWARD (strip convex) : {implied_fwd:,.4f}")
print(f"Implied carry  convex-aware    : {carry_cvx:.4%}   (naive: {implied_carry:.4%})")
print(f"Implied borrow convex-aware    : {b_cvx:.4%}   (naive: {implied_b:.4%})")

Market quote (as FUTURES)      : 5,520.0000
Convexity c                    : +0.1245 bp
Implied FORWARD (strip convex) : 5,519.9313
Implied carry  convex-aware    : 1.4469%   (naive: 1.4519%)
Implied borrow convex-aware    : 1.5531%   (naive: 1.5481%)


In [20]:
def imply_carry_from_futures(market_fut, S0, r, q, T_, b_assumed=0.0, use_convexity=True,
                             equity_vol=sigma, rate_vol=sigma_r, corr=rho_Sr, mean_rev=a_mr):
    """Invert a futures quote to an implied carry. With use_convexity=True (default) the
    Hull-White convexity is stripped first (Fwd = Fut*exp(-c)). Set use_convexity=False to
    skip it and treat the quote directly as a forward -- simpler, and fine for short tenors
    where c is sub-basis-point. (Setting corr=0 or rate_vol=0 also makes c vanish.)"""
    c = (futures_convexity_logadj(a(equity_vol), a(rate_vol), a(corr), a(T_), a(mean_rev)).item()
         if use_convexity else 0.0)
    fwd = market_fut * np.exp(-c)
    carry = np.log(fwd / S0) / T_                     # closed-form inverse of S0*exp(carry*T)
    return {"convexity_bp": c*1e4, "implied_forward": fwd, "implied_carry": carry,
            "implied_borrow_given_q": r - q - carry, "implied_div_given_b": r - b_assumed - carry}

# Round-trip: model futures price -> implied borrow, with and without the convexity correction.
print(f"{'T':>5} | {'conv bp':>8} | {'b (true)':>9} | {'b convex-aware':>15} | {'b naive':>9}")
print("-" * 60)
for t in [0.25, 0.5, 1.0, 2.0, 5.0]:
    fwd_t = future_fair_price(a(S0), a(r), a(q), a(b), a(t)).item()
    c_t   = futures_convexity_logadj(a(sigma), a(sigma_r), a(rho_Sr), a(t), a(a_mr)).item()
    fut_t = fwd_t * np.exp(c_t)                       # the model's futures price
    b_aware = imply_carry_from_futures(fut_t, S0, r, q, t)["implied_borrow_given_q"]
    b_naive = r - q - np.log(fut_t / S0) / t          # ignore convexity (treat future as forward)
    print(f"{t:5.2f} | {c_t*1e4:8.4f} | {b:9.4%} | {b_aware:15.4%} | {b_naive:9.4%}")
assert abs(imply_carry_from_futures(fut_t, S0, r, q, 5.0)["implied_borrow_given_q"] - b) < 1e-9
b_off = imply_carry_from_futures(fut_t, S0, r, q, 5.0, use_convexity=False)["implied_borrow_given_q"]
print(f"\nuse_convexity=False at T=5 -> implied borrow {b_off:.4%} (matches the naive column).")
print("Convexity-aware inversion recovers the true borrow at every maturity. ✓")

    T |  conv bp |  b (true) |  b convex-aware |   b naive
------------------------------------------------------------
 0.25 |   0.1245 |   0.2500% |         0.2500% |   0.2450%
 0.50 |   0.4959 |   0.2500% |         0.2500% |   0.2401%
 1.00 |   1.9671 |   0.2500% |         0.2500% |   0.2303%
 2.00 |   7.7399 |   0.2500% |         0.2500% |   0.2113%
 5.00 |  46.0813 |   0.2500% |         0.2500% |   0.1578%

use_convexity=False at T=5 -> implied borrow 0.1578% (matches the naive column).
Convexity-aware inversion recovers the true borrow at every maturity. ✓


---
*Built on `AnonymousJY/mkt-depth-n-resiliency` (MIT). The library implements the methods in
*Systematic Liquidity Risk Management: A Novel Perspective on Derivatives*
([SSRN 5454874](https://dx.doi.org/10.2139/ssrn.5454874)). For illustration only — not
investment advice.*